# Chapter 04. Naive Bayes로 도서 카테고리 분류하기

Chapter 03에서는 상품명을 CountVectorizer와 TF-IDF로 숫자 벡터로 바꿨다.
이번 Chapter에서는 그 벡터를 머신러닝 모델에 넣어 **도서 제목으로 분야를 예측**한다.

> 도서 제목 -> TF-IDF -> Multinomial Naive Bayes -> 예상 분야

이번 Chapter의 목표는 최고 성능을 만드는 것이 아니다. 다음 순서를 정확히 이해하는 것이 핵심이다.

> **원본 데이터를 먼저 train/test로 나누고, TF-IDF는 train 데이터에만 fit한다.**

테스트 데이터의 정보를 학습 과정에 미리 사용하면 평가를 신뢰하기 어렵다.

## 최종 결과물

```
chapter04.ipynb
chapter04_predictions.csv
chapter04_misclassified.csv
```

---
# 1. 학습 목표

실습이 끝나면 다음을 설명할 수 있어야 한다.

- 지도학습과 분류의 의미
- 입력 X와 정답 y
- train/test를 나누는 이유
- `fit()`과 `transform()`의 차이
- TF-IDF를 train에만 fit해야 하는 이유
- Multinomial Naive Bayes의 역할
- accuracy와 분야별 평가 지표
- 오분류 사례를 확인하는 이유
- 새로운 제목을 예측하는 방법

## 전체 흐름

```
데이터 준비 -> X, y 정의 -> train/test 분리 -> train에 TF-IDF fit
-> train/test transform -> Naive Bayes 학습 -> test 예측
-> 성능과 오분류 확인 -> 새 제목 예측
```

---
# 실습 1. 라이브러리 준비

## 해야 할 일

분류에 필요한 pandas, scikit-learn 기능을 불러온다.
필요한 패키지가 없다면 가상환경에서 설치한다.

```
python -m pip install pandas scikit-learn
```

## 프롬프트 예시

```
도서 제목으로 분야를 예측하는 텍스트 분류를 하려고 합니다.
pandas, train_test_split, TfidfVectorizer, MultinomialNB,
그리고 accuracy_score, classification_report, confusion_matrix를 불러오는
import 코드를 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

print("라이브러리 준비 완료")

### 작업 폴더를 맞춘다

이 저장소의 `.vscode/settings.json`에 `"jupyter.notebookFileRoot": "${workspaceFolder}"`가 있어서,
Notebook을 실행하면 작업 폴더가 노트북이 있는 곳이 아니라 **저장소 루트**가 된다.
그래서 파일 이름만 적으면 `FileNotFoundError`가 난다.
아래 셀에서 작업 폴더를 `book-text-ml`로 맞춘 뒤 진행한다.

## 프롬프트 예시

```
VS Code에서 Jupyter Notebook을 쓰고 있습니다.
설정 때문에 작업 폴더가 노트북이 있는 곳이 아니라 저장소 루트가 됩니다.
노트북이 들어 있는 book-text-ml 폴더를 찾아 그쪽으로 이동하고 싶습니다.
폴더가 저장소 루트에 있을 수도 있고 notebooks 안에 있을 수도 있습니다.
마지막에 현재 작업 폴더와 데이터 파일 존재 여부를 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
import os
from pathlib import Path

# 노트북이 있는 book-text-ml 폴더를 찾아 그쪽으로 이동한다.
# 폴더가 저장소 루트에 있든 notebooks 안에 있든 모두 찾는다.
if Path.cwd().name != "book-text-ml":
    for candidate in [Path("book-text-ml"), *Path(".").glob("*/book-text-ml")]:
        if candidate.is_dir():
            os.chdir(candidate)
            break

print("작업 폴더:", Path.cwd())
print("데이터 파일 있음:", Path("book_bestseller_clean.csv").exists())

---
# 실습 2. 데이터 불러오기

## 해야 할 일

Chapter 01에서 만든 `book_bestseller_clean.csv`를 사용한다.
이번 분류에서 필요한 컬럼은 두 개다.

| 컬럼 | 역할 |
|---|---|
| 상품명 | 모델 입력 |
| 분야 | 모델이 맞혀야 할 정답 |

## 프롬프트 예시

```
Python과 pandas를 처음 배우고 있습니다.
현재 Notebook과 같은 폴더에 book_bestseller_clean.csv 파일이 있습니다.
utf-8-sig 인코딩으로 df_books라는 DataFrame으로 불러오고,
데이터 크기, 컬럼 목록, 상품명과 분야 두 컬럼의 앞 10개를 확인하고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
DATA_PATH = "book_bestseller_clean.csv"
df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())
df_books[["상품명", "분야"]].head(10)

---
# 실습 3. 모델링 데이터 정리하기

## 해야 할 일

필요한 두 컬럼만 복사하고, 입력과 정답에 빈 값이 없도록 정리한다.
**모델링 전에 입력과 정답에 빈 값이 없는지 확인하는 습관**이 중요하다.

## 프롬프트 예시

```
DataFrame df_books에서 상품명과 분야 두 컬럼만 복사해 df_model을 만들고 싶습니다.
두 컬럼 모두 결측치는 빈 문자열로 바꾸고, 문자열로 통일하고, 앞뒤 공백을 제거한 뒤
상품명이나 분야가 비어 있는 행은 빼고 인덱스를 다시 매겨 주세요.
마지막에 데이터 크기를 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
df_model = df_books[["상품명", "분야"]].copy()

for col in ["상품명", "분야"]:
    df_model[col] = (
        df_model[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

df_model = df_model[
    (df_model["상품명"] != "") &
    (df_model["분야"] != "")
].reset_index(drop=True)

print("모델링 데이터 크기:", df_model.shape)

---
# 실습 4. 분야 분포 확인하기

## 해야 할 일

분야별 데이터 수가 크게 다르면 **클래스 불균형**이 있을 수 있다.
무조건 데이터를 삭제하거나 수를 맞추기보다 먼저 분포를 확인하고,
이후 분야별 성능과 오분류를 함께 살펴본다.

## 프롬프트 예시

```
DataFrame df_model의 분야 컬럼이 몇 종류인지 알고 싶습니다.
분야 종류 수를 출력하고 분야별 도서 수 상위 20개를 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
print("분야 종류 수:", df_model["분야"].nunique())
df_model["분야"].value_counts().head(20)

### stratify를 쓰기 전에 반드시 확인할 것

실습 6에서 `stratify=y`를 쓰려면 **모든 분야에 최소 2권**이 있어야 한다.
1권뿐인 분야는 train과 test에 나눠 담을 수 없어서 오류가 난다.

## 프롬프트 예시

```
train_test_split에서 stratify=y를 쓰려고 합니다.
DataFrame df_model에서 도서가 1권뿐인 분야가 있는지 확인하고 싶습니다.
분야별 도서 수를 구해서 2권 미만인 분야만 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
class_counts = df_model["분야"].value_counts()

print("1권뿐인 분야:")
print(class_counts[class_counts < 2])

## 프롬프트 예시

```
분야별 도서 수가 든 Series class_counts가 있습니다.
도서가 1권뿐인 분야의 실제 제목이 무엇인지 df_model에서 찾아 보고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 1권뿐인 분야의 실제 제목을 확인한다
rare_fields = class_counts[class_counts < 2].index
df_model[df_model["분야"].isin(rare_fields)]

### 확인 결과

1권뿐인 분야가 **6개** 있었다. 실제 제목을 보면 다음과 같다.

- `강연`, `가요`, `필기구` -> Chapter 01에서 발견한 **도서가 아닌 상품**이다.
- `인문/사회`, `ELT/수험서`, `과학/기술` -> 비슷한 이름의 큰 분야(`인문`, `외국어` 등)가 따로 있다.
  분류 체계가 섞여 들어온 것으로 보인다.

그리고 한 가지가 더 있다.
Chapter 01에서 분야가 비어 있던 책을 `미분류`로 채웠는데,
여기서는 이것이 **정답 라벨**로 들어간다.
"미분류"를 맞히도록 학습시키는 것은 의미가 없다.

### 이번 Chapter의 처리 기준

| 대상 | 처리 | 이유 |
|---|---|---|
| 1권뿐인 분야 6개 | 제외 | train/test에 나눌 수 없고, 1권으로는 학습도 평가도 할 수 없다 |
| `미분류` | 제외 | 실제 분야가 아니라 결측치를 채운 값이다 |

블로그의 권고대로 `stratify=None`으로 바꿔서 오류를 피하지 않고,
**먼저 확인한 뒤 근거를 남기고 처리**한다.
제외 전후 결과 비교는 뒤의 보충 분석에서 확인한다.

## 프롬프트 예시

```
DataFrame df_model에서 두 가지를 빼고 싶습니다.
1. 도서가 1권뿐인 분야 (stratify로 나눌 수 없음)
2. 분야가 '미분류'인 행 (결측치를 채운 값이라 정답으로 쓸 수 없음)
빼기 전과 후의 도서 수, 뺀 개수, 남은 분야 수,
그리고 가장 적은 분야의 도서 수를 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
before = len(df_model)

keep_fields = class_counts[class_counts >= 2].index
df_model = df_model[
    df_model["분야"].isin(keep_fields) &
    (df_model["분야"] != "미분류")
].reset_index(drop=True)

print("정리 전:", before, "권")
print("정리 후:", len(df_model), "권  (", before - len(df_model), "권 제외 )")
print("남은 분야 수:", df_model["분야"].nunique())
print("가장 적은 분야의 도서 수:", df_model["분야"].value_counts().min())

---
# 실습 5. X와 y 정의하기

## 해야 할 일

지도학습은 **입력과 정답이 함께 있는 데이터**로 관계를 학습한다.

```
상품명: 파이썬 데이터 분석 입문    ->  X = "파이썬 데이터 분석 입문"
분야:   컴퓨터/IT                ->  y = "컴퓨터/IT"
```

이번 문제는 여러 분야 중 하나를 예측하므로 **분류 문제**다.

## 프롬프트 예시

```
DataFrame df_model에서 상품명을 입력 X로, 분야를 정답 y로 나누고 싶습니다.
X와 y의 개수, 그리고 첫 번째 X와 y 예시를 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
X = df_model["상품명"]
y = df_model["분야"]

print("X 개수:", len(X))
print("y 개수:", len(y))
print()
print("X 예시:", X.iloc[0])
print("y 예시:", y.iloc[0])

---
# 실습 6. train/test 분리하기

## 해야 할 일

학습에 사용한 데이터로만 성능을 확인하면 새로운 데이터에서도 잘 작동하는지 판단하기 어렵다.

```
train = 공부할 문제
test  = 처음 보는 시험 문제
```

| 옵션 | 의미 |
|---|---|
| `test_size=0.2` | 약 20%를 평가에 사용 |
| `random_state=42` | 같은 분할을 재현 |
| `stratify=y` | 분야 비율을 가능한 비슷하게 유지 |

## 프롬프트 예시

```
입력 X와 정답 y가 있습니다.
train_test_split으로 20%를 test로 나누고 싶습니다.
같은 결과가 나오게 random_state=42를 쓰고,
분야 비율이 train과 test에서 비슷하도록 stratify=y를 써 주세요.
train과 test의 크기를 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

## 프롬프트 예시

```
train_test_split에서 stratify=y를 썼습니다.
분야 비율이 train과 test에서 정말 비슷한지 확인하고 싶습니다.
y_train과 y_test의 분야별 비율을 한 표에 나란히 놓고
소수점 3자리로 상위 8개만 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# stratify가 제대로 됐는지: 상위 분야의 비율이 train과 test에서 비슷한지 확인
ratio = pd.DataFrame({
    "train 비율": y_train.value_counts(normalize=True),
    "test 비율": y_test.value_counts(normalize=True),
}).round(3)

ratio.head(8)

---
# 실습 7. 가장 중요한 원칙 — split을 먼저 한다

### 잘못된 순서
```
전체 데이터에 TF-IDF fit -> train/test 분리 -> 모델 학습과 평가
```

### 올바른 순서
```
원본 텍스트 -> train/test 분리 -> train에 TF-IDF fit -> test는 transform만 수행 -> 모델 학습과 평가
```

전체 데이터에 TF-IDF를 먼저 fit하면 **test에 있는 단어 정보가 학습 과정에 들어갈 수 있다.**
이를 **데이터 누수(data leakage)** 의 한 형태로 볼 수 있다.

---
# 실습 8. fit과 transform 이해하기

| 메서드 | 하는 일 |
|---|---|
| `fit` | train 데이터에서 단어 사전과 IDF 규칙을 **학습** |
| `transform` | 이미 학습한 규칙으로 문장을 **숫자로 변환** |

따라서 **train에는 `fit_transform()`, test에는 `transform()`** 을 사용한다.

---
# 실습 9. TF-IDF 변환하기

## 해야 할 일

train에만 fit하고, test는 transform만 한다.

train과 test의 **행 수는 다르지만 열 수는 같아야 한다.** 같은 Vectorizer의 단어 공간을 쓰기 때문이다.

test에만 등장하는 새로운 단어가 vocabulary에 없을 수 있는데, **이것은 정상**이다.
새 데이터가 들어올 때마다 Vectorizer를 다시 fit하면 학습 때와 다른 좌표계를 만들게 된다.

## 프롬프트 예시

```
텍스트 X_train과 X_test가 있습니다.
TfidfVectorizer를 train에만 fit하고, test는 transform만 하고 싶습니다.
test에 fit_transform을 쓰면 안 됩니다.
두 행렬의 크기와, 열 수가 같은지 여부를 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF:", X_train_tfidf.shape)
print("Test TF-IDF :", X_test_tfidf.shape)
print("열 수가 같은가?:", X_train_tfidf.shape[1] == X_test_tfidf.shape[1])

## 프롬프트 예시

```
train에만 fit한 TF-IDF로 test를 변환한 X_test_tfidf가 있습니다.
test 제목 중에서 train 단어 사전에 있는 단어가 하나도 없어
벡터가 전부 0인 제목이 몇 개인지 세고, 그런 제목 5개를 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# test 제목 중 train 단어 사전에 있는 단어가 하나도 없는 제목은 몇 개인가
empty_rows = (X_test_tfidf.getnnz(axis=1) == 0)

print("단어 사전에 있는 단어가 0개인 test 제목:", int(empty_rows.sum()), "/", len(X_test))
print()
for t in X_test[empty_rows].head(5):
    print("  -", t)

위 제목들은 **train에서 한 번도 본 적 없는 단어로만** 이루어져 있다.
TF-IDF 벡터가 모두 0이 되므로 모델은 제목에서 아무 정보도 얻지 못한다.
이런 제목을 모델이 어떻게 예측하는지는 실습 17에서 확인한다.

---
# 실습 10. Multinomial Naive Bayes 이해하기

초보자 단계에서는 Naive Bayes를 이렇게 이해하면 충분하다.

```
어떤 단어 패턴이 어떤 분야에서 자주 나타나는지 학습
        ↓
새 제목의 단어 패턴을 보고 가능성이 높은 분야를 예측
```

Multinomial Naive Bayes는 구조가 단순하고 학습이 빠르며,
**희소한 텍스트 벡터와 함께 쓰기 좋아** baseline 텍스트 분류 모델로 적합하다.

## 프롬프트 예시

```
텍스트 분류용으로 Multinomial Naive Bayes 모델을 만들고 싶습니다.
기본 설정으로 model이라는 변수에 만들어 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
model = MultinomialNB()
print(model)

---
# 실습 11. 모델 학습과 예측

## 해야 할 일

1. train 데이터로 모델을 학습한다
2. test 데이터를 예측하고 정확도를 출력한다
3. 실제 정답과 모델 예측값을 샘플로 비교해 본다
4. 전체 결과를 한 표로 만든다

## 11-1. train 데이터로 모델 학습

`fit()`에는 **train 데이터만** 넣는다.
학습이 끝나면 모델이 무엇을 배웠는지 간단히 확인한다.

- 몇 권으로, 몇 개 분야를, 몇 개 단어로 학습했는가
- 분야별 **사전 확률**(train에서 각 분야가 차지하는 비율)은 어떤가

사전 확률은 제목에서 단서를 찾지 못했을 때 모델이 기대는 값이라 꼭 확인해 둔다.

## 프롬프트 예시

```
TF-IDF로 변환한 train 데이터 X_train_tfidf와 정답 y_train이 있습니다.
MultinomialNB 모델 model을 train 데이터로만 학습시키고,
학습에 사용한 도서 수, 학습한 분야 수, 사용한 단어 수를 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
model.fit(X_train_tfidf, y_train)

print("모델 학습 완료")
print("  학습에 사용한 도서 수:", X_train_tfidf.shape[0], "권")
print("  학습한 분야 수     :", len(model.classes_), "개")
print("  사용한 단어 수     :", X_train_tfidf.shape[1], "개")

## 프롬프트 예시

```
학습이 끝난 MultinomialNB model이 있습니다.
분야별 사전 확률(train에서 각 분야가 차지하는 비율)을 보고 싶습니다.
model.class_log_prior_는 로그 값이니 원래 확률로 바꿔서
큰 순서로 상위 5개를 소수점 3자리로 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 분야별 사전 확률 상위 5개 (train에서 각 분야가 차지하는 비율)
import numpy as np

prior = pd.Series(np.exp(model.class_log_prior_), index=model.classes_)
prior.sort_values(ascending=False).head(5).round(3)

`소설`의 사전 확률이 0.194로 가장 높다.
제목에서 분야를 알 수 있는 단어를 찾지 못하면 모델은 이 값에 기대기 때문에,
**애매한 제목은 소설로 예측될 가능성이 높다**는 것을 미리 짐작할 수 있다.

## 11-2. 예측 및 결과 정확도 출력

`predict()`로 test를 예측하고 정확도를 출력한다.

같은 모델로 **train 데이터도 예측**해서 두 정확도를 비교한다.

| 비교 결과 | 의미 |
|---|---|
| train과 test가 비슷하게 높다 | 잘 학습했다 |
| train만 높고 test가 낮다 | train을 외웠다 (과적합) |
| 둘 다 낮다 | 제목만으로 구분할 정보가 부족하다 |

## 프롬프트 예시

```
학습된 model로 test 데이터를 예측하고 정확도를 출력하고 싶습니다.
같은 모델로 train 데이터도 예측해서 train 정확도와 test 정확도를 나란히 비교하고,
두 값의 차이도 출력해 주세요.
test에서 맞힌 개수와 틀린 개수도 함께 보여 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
y_pred = model.predict(X_test_tfidf)
y_train_pred = model.predict(X_train_tfidf)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_pred)

print(f"train 정확도: {train_acc:.4f}   (공부한 문제를 다시 풀었을 때)")
print(f"test  정확도: {test_acc:.4f}   (처음 보는 문제를 풀었을 때)")
print(f"차이        : {train_acc - test_acc:.4f}")
print()
print("test 맞힌 개수:", int((y_test.values == y_pred).sum()), "/", len(y_test))
print("test 틀린 개수:", int((y_test.values != y_pred).sum()), "/", len(y_test))

### 결과 해석

train 정확도 0.6011, test 정확도 0.4130으로 **둘 다 높지 않다.**

train조차 60%밖에 못 맞힌다는 것은, 모델이 공부한 문제도 40%는 틀린다는 뜻이다.
과적합(외우기)보다는 **제목 안에 분야를 가려낼 단서가 부족한 것**이 더 큰 원인이다.
도서 제목은 평균 3~4개 단어로 짧고, 단어의 70% 이상이 한 번만 등장하기 때문이다(Chapter 03).

## 11-3. 실제 정답과 모델 예측값 샘플 5개 비교

숫자로 된 정확도만 보면 모델이 **어떻게** 맞히고 틀리는지 알 수 없다.
실제 제목을 몇 개 골라 정답과 예측을 나란히 본다.

## 프롬프트 예시

```
test 제목 X_test, 실제 분야 y_test, 예측 결과 y_pred가 있습니다.
앞에서부터 5개를 골라 제목, 실제 분야, 예측 분야를 출력하고
맞았으면 'O 정답', 틀렸으면 'X 오답'으로 표시해 주세요.
보기 쉽게 샘플마다 한 줄씩 띄워 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# test 앞에서부터 5개를 비교한다
for i in range(5):
    title = X_test.iloc[i]
    actual = y_test.iloc[i]
    predicted = y_pred[i]
    mark = "O 정답" if actual == predicted else "X 오답"

    print(f"[{i + 1}] {title}")
    print(f"     실제: {actual}  /  예측: {predicted}  ->  {mark}")
    print()

## 프롬프트 예시

```
test 제목, 실제 분야, 예측 분야로 비교용 DataFrame을 만들고 싶습니다.
맞힌 예 5개와 틀린 예 5개를 각각 따로 표로 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 맞힌 예와 틀린 예를 각각 5개씩 따로 본다
compare = pd.DataFrame({
    "상품명": X_test.values,
    "실제_분야": y_test.values,
    "예측_분야": y_pred,
})

print("=== 맞힌 예 5개 ===")
print(compare[compare["실제_분야"] == compare["예측_분야"]].head(5).to_string(index=False))
print()
print("=== 틀린 예 5개 ===")
print(compare[compare["실제_분야"] != compare["예측_분야"]].head(5).to_string(index=False))

### 샘플에서 보이는 것

**맞힌 예**는 제목에 분야가 드러나는 단어가 있었다.

```
2027 공단기 심슨 문법 300제      -> 취업/수험서  (공단기, 2027)
설민석의 한국사 대모험 38       -> 어린이(초등) (대모험)
```

**틀린 예**는 거의 모두 `소설`로 예측됐다.

```
인플레이션은 누구를 부자로 만드는가   실제: 경제/경영  -> 예측: 소설
[비욘드오리진] ABC착즙액 (50ml*20포)  실제: 홈/라이프  -> 예측: 소설
```

앞에서 확인한 대로, 단서가 약하면 사전 확률이 가장 높은 `소설`로 기울었다.
두 번째 예는 **도서가 아닌 상품**이라 제목만으로는 분야를 알 수 없는 경우다.

## 11-4. 전체 결과를 한 표로 만들기

이후 오분류 확인과 파일 저장에 쓰기 위해 전체 결과를 DataFrame으로 만든다.

## 프롬프트 예시

```
test 제목 X_test, 실제 분야 y_test, 예측 결과 y_pred가 있습니다.
상품명, 실제_분야, 예측_분야 세 컬럼을 가진 DataFrame result로 만들고
앞의 20개를 보여 주세요.
X_test와 y_test는 인덱스를 0부터 다시 매겨 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
result = pd.DataFrame({
    "상품명": X_test.reset_index(drop=True),
    "실제_분야": y_test.reset_index(drop=True),
    "예측_분야": y_pred,
})
result.head(20)

---
# 실습 12. Accuracy 확인하기

## 해야 할 일

Accuracy는 **전체 test 데이터 중 올바르게 예측한 비율**이다.

하지만 클래스별 데이터 수가 다르면 accuracy 하나만으로 모델을 평가하기 어렵다.
그래서 **"아무것도 배우지 않고 가장 많은 분야만 찍었을 때"의 정확도(기준선)** 와 함께 본다.

## 프롬프트 예시

```
실제 분야 y_test와 예측 y_pred가 있습니다.
accuracy_score로 정확도를 소수점 4자리까지 출력하고,
몇 개 중 몇 개를 맞혔는지도 함께 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("맞힌 개수:", int((y_test.values == y_pred).sum()), "/", len(y_test))

## 프롬프트 예시

```
모델 정확도가 좋은 건지 판단할 기준이 필요합니다.
train에서 가장 많은 분야 하나만 계속 찍었을 때의 test 정확도(기준선)를 구하고,
모델 정확도가 기준선의 몇 배인지 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 기준선: train에서 가장 많은 분야 하나만 계속 찍었을 때
most_common = y_train.value_counts().index[0]
baseline = (y_test == most_common).mean()

print("가장 많은 분야:", most_common)
print(f"기준선 Accuracy: {baseline:.4f}")
print(f"모델 Accuracy  : {accuracy:.4f}")
print(f"기준선 대비    : {accuracy / baseline:.1f}배")

---
# 실습 13. Classification Report 확인하기

## 해야 할 일

분야별 precision, recall, F1-score를 확인한다.

| 지표 | 의미 |
|---|---|
| precision | 해당 분야라고 **예측한 것 중** 실제로 맞은 비율 |
| recall | 실제 해당 분야 **중 모델이 찾아낸** 비율 |
| F1-score | precision과 recall을 함께 고려한 지표 |

모든 숫자를 외우기보다 **분야에 따라 성능 차이가 있는지**를 살펴본다.

## 프롬프트 예시

```
실제 분야 y_test와 예측 y_pred가 있습니다.
분야별 precision, recall, F1-score를 한 번에 보고 싶습니다.
예측이 한 번도 안 된 분야가 있어도 경고가 나지 않도록 zero_division=0을 써 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
report = classification_report(
    y_test,
    y_pred,
    zero_division=0,
)
print(report)

---
# 실습 14. Confusion Matrix 확인하기

## 해야 할 일

Confusion Matrix는 **어떤 분야가 어떤 분야로 자주 잘못 분류되는지** 확인하는 데 도움이 된다.

분야가 31개나 되어 전체 행렬은 읽기 어렵다.
그래서 행렬을 만든 뒤, **자주 헷갈린 분야 쌍**만 뽑아서 본다.

## 프롬프트 예시

```
학습된 model, 실제 y_test, 예측 y_pred가 있습니다.
confusion matrix를 만들어 행과 열 이름이 분야인 DataFrame으로 보고 싶습니다.
분야 순서는 model.classes_를 써 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
labels = model.classes_

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels,
)

df_cm = pd.DataFrame(
    cm,
    index=labels,
    columns=labels,
)
df_cm

## 프롬프트 예시

```
confusion matrix DataFrame df_cm이 있습니다. 분야가 많아서 전체를 읽기 어렵습니다.
정답(대각선)을 뺀 나머지 중 많이 틀린 '실제_분야 -> 예측_분야' 쌍을
건수가 많은 순서로 10개만 보고 싶습니다.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 대각선(정답)을 뺀 나머지 중 많이 틀린 쌍만 뽑는다
pairs = (
    df_cm.stack()
    .rename_axis(["실제_분야", "예측_분야"])
    .reset_index(name="건수")
)
pairs = pairs[(pairs["실제_분야"] != pairs["예측_분야"]) & (pairs["건수"] > 0)]
pairs.sort_values("건수", ascending=False).head(10).reset_index(drop=True)

---
# 실습 15. 오분류 사례 확인하기

## 해야 할 일

틀린 제목을 직접 읽어보면서 다음을 질문한다.

- 제목만으로 분야를 판단하기 어려운가?
- 여러 분야에서 사용할 수 있는 단어인가?
- 학습 데이터가 부족한 분야인가?
- 분야 라벨이 지나치게 세분화되어 있는가?

**틀린 사례를 보는 것이 모델 개선의 출발점이다.**

## 프롬프트 예시

```
예측 결과가 든 DataFrame result가 있습니다.
실제_분야와 예측_분야가 다른 행만 골라 misclassified로 만들고,
오분류 개수와 앞의 20개를 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
misclassified = result[
    result["실제_분야"] != result["예측_분야"]
].copy()

print("오분류 수:", len(misclassified), "/", len(result))
misclassified.head(20)

---
# 실습 15-1. 오분류 원인 분석

## 해야 할 일

오분류 사례를 몇 개 읽어보는 것만으로는 **어떤 원인이 얼마나 많은지** 알 수 없다.
틀린 108건을 원인별로 나누고, 원인마다 몇 건인지 숫자로 확인한다.

확인할 것은 다음과 같다.

- 틀린 예측이 어느 분야로 몰렸는가
- 제목에서 **단어 사전에 있는 단어가 몇 개**였는가 (모델이 실제로 쓸 수 있었던 단서의 양)
- 원인별로 몇 건씩인가
- 맞힌 것과 틀린 것의 **확신도**는 어떻게 다른가

## 프롬프트 예시

```
오분류만 모은 DataFrame misclassified가 있습니다.
틀리게 예측한 분야가 어디로 몰렸는지 상위 5개를 보고,
그중 '소설'로 잘못 예측한 건수와 비율을 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 1) 틀린 예측이 어느 분야로 몰렸는가
print("오분류 수:", len(misclassified))
print()
print("틀리게 예측한 분야 상위 5개:")
print(misclassified["예측_분야"].value_counts().head(5))
print()

to_novel = int((misclassified["예측_분야"] == "소설").sum())
print(f"소설로 잘못 예측한 비율: {to_novel} / {len(misclassified)} ({to_novel / len(misclassified) * 100:.0f}%)")

## 프롬프트 예시

```
예측 결과 DataFrame result와 test TF-IDF 행렬 X_test_tfidf가 있습니다.
제목마다 TF-IDF 단어 사전에 있던 단어가 몇 개인지(getnnz)와 정답 여부를 컬럼으로 붙이고,
단어 수를 0개, 1개, 2개, 3개 이상으로 나눠 구간별 제목 수와 정확도를 보고 싶습니다.
원본 result는 바꾸지 말고 복사본 diag에서 작업해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 2) 제목마다 "단어 사전에 있던 단어 수"를 붙이고, 단어 수별 정확도를 본다
diag = result.copy()
diag["사전단어수"] = X_test_tfidf.getnnz(axis=1)
diag["정답여부"] = diag["실제_분야"] == diag["예측_분야"]

bucket = pd.cut(diag["사전단어수"], bins=[-1, 0, 1, 2, 100],
                labels=["0개", "1개", "2개", "3개 이상"])

(diag.groupby(bucket, observed=True)["정답여부"]
     .agg(["count", "mean"])
     .round(3)
     .rename(columns={"count": "제목 수", "mean": "정확도"}))

사전에 있는 단어가 **3개 이상**인 제목은 정확도가 0.613이지만,
**1~2개**인 제목은 0.233~0.300으로 뚝 떨어진다.

`0개`가 0.381로 1~2개보다 높은 것은 모델이 잘해서가 아니다.
단어 정보가 전혀 없으면 모델은 사전 확률이 가장 높은 `소설`로 찍는데,
그중 실제로 소설인 책이 섞여 있어 **운 좋게 맞은 것**이다.

## 프롬프트 예시

```
오분류 원인을 규칙으로 나눠 보고 싶습니다.
1. 실제 분야가 홈/라이프, 책향, 교보문고 굿즈 -> 도서가 아닌 상품
2. 사전에 있는 단어 0개
3. 사전에 있는 단어 1~2개
4. 단어가 3개 이상인데 틀림
diag에 사전에 있던 단어 목록 컬럼도 추가하고,
틀린 행에 원인 컬럼을 붙여 원인별 건수를 보여 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 3) 오분류를 원인별로 나눈다
feature_names = tfidf.get_feature_names_out()
diag["사전단어"] = [
    feature_names[X_test_tfidf.getrow(i).indices].tolist()
    for i in range(X_test_tfidf.shape[0])
]

non_book_fields = ["홈/라이프", "책향", "교보문고 굿즈"]   # Chapter 01에서 확인한 비도서 분야

def find_cause(row):
    if row["실제_분야"] in non_book_fields:
        return "1. 도서가 아닌 상품"
    if row["사전단어수"] == 0:
        return "2. 사전에 있는 단어 0개"
    if row["사전단어수"] <= 2:
        return "3. 단서 단어 1~2개"
    return "4. 단어는 충분하나 다른 분야로 끌림"

wrong = diag[~diag["정답여부"]].copy()
wrong["원인"] = wrong.apply(find_cause, axis=1)

wrong["원인"].value_counts().sort_index()

## 프롬프트 예시

```
원인 컬럼이 붙은 오분류 DataFrame wrong이 있습니다.
원인별로 대표 사례 3개씩, 제목·실제 분야·예측 분야·사전에 있던 단어를 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 4) 원인별 대표 사례 3개씩
for cause, group in wrong.groupby("원인"):
    print(f"=== {cause} ({len(group)}건) ===")
    for _, row in group.head(3).iterrows():
        words = row["사전단어"] if row["사전단어"] else "없음"
        print(f"  {row['상품명'][:38]}")
        print(f"      실제 {row['실제_분야']} -> 예측 {row['예측_분야']}   | 사전에 있던 단어: {words}")
    print()

## 프롬프트 예시

```
학습된 model과 X_test_tfidf, 정답 여부가 든 diag가 있습니다.
각 제목의 가장 높은 예측 확률을 '확신도'로 붙이고,
맞힌 것과 틀린 것의 평균 확신도를 비교해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 5) 맞힌 것과 틀린 것의 확신도(가장 높은 예측 확률) 비교
diag["확신도"] = model.predict_proba(X_test_tfidf).max(axis=1)

diag.groupby("정답여부")["확신도"].mean().round(3)

## 오분류 원인 분석 결과

틀린 108건 중 **99건(92%)이 `소설`로 예측**되었다. 원인을 나누면 다음과 같다.

| 원인 | 건수 | 대표 사례 |
|---|---|---|
| 1. 도서가 아닌 상품 | 5 | `The Scent of Page : 디퓨저 300ML` (책향 -> 소설) |
| 2. 사전에 있는 단어 0개 | 25 | `인플레이션은 누구를 부자로 만드는가` (경제/경영 -> 소설) |
| 3. 단서 단어 1~2개 | 56 | `기억 전달자` (청소년 -> 소설) |
| 4. 단어는 충분하나 다른 분야로 끌림 | 22 | `미움받을 용기(200만 부 기념 스페셜 에디션)` (인문 -> 소설) |

### 원인별 해석

**2번 — 사전에 있는 단어 0개 (25건)**
`인플레이션은 누구를 부자로 만드는가`는 사전에 `부자`가 있는데도 단어가 0개로 잡혔다.
기본 TfidfVectorizer는 조사를 떼지 않아 `부자로`를 `부자`와 **다른 단어**로 보기 때문이다.
한국어 제목에서 이런 일이 흔하다는 것이 가장 큰 구조적 원인이다.

**3번 — 단서 단어 1~2개 (56건, 가장 많음)**
`싫어하는 사람을 사라지게 하는 방법`에서 사전에 있던 단어는 `사람을` 하나뿐이었다.
단어가 한두 개면 그 단어가 **어느 분야에서나 쓰이는 일반적인 단어**일 가능성이 높아
분야를 가려내지 못하고 사전 확률로 기운다.

**4번 — 다른 분야로 끌림 (22건)**
`미움받을 용기(200만 부 기념 스페셜 에디션)`에서 사전에 있던 단어는
`기념`, `스페셜`, `에디션` 세 개였는데 **셋 다 책 내용이 아니라 판매용 문구**였다.
이 단어들은 train에서 소설 개정판에 자주 붙어 있어서 소설 쪽으로 끌려갔다.
제목 자체(`미움받을 용기`)는 사전에 없었다.

**1번 — 도서가 아닌 상품 (5건)**
`디퓨저`, `담요`, `착즙액`은 제목만으로 분야를 맞히기 어렵다.
Chapter 01에서 확인한 비도서 상품이 분류 결과에도 그대로 영향을 줬다.

### 확신도

맞힌 것의 평균 확신도는 0.281, 틀린 것은 0.198이었다.
**맞힌 경우에도 확신도가 0.3이 안 된다.** 31개 분야 중 하나를 고르는데
가장 높은 확률이 0.28이라는 것은, 모델이 대부분의 제목에서 **확신 없이 고르고 있다**는 뜻이다.

---
# 실습 16. 예측 결과 저장하기

저장한 파일은 **실제로 다시 열어** 한글과 컬럼이 정상인지 확인한다.

## 프롬프트 예시

```
예측 결과 result와 오분류 결과 misclassified가 있습니다.
chapter04_predictions.csv와 chapter04_misclassified.csv로 저장하고 싶습니다.
utf-8-sig 인코딩을 쓰고 인덱스는 저장하지 말아 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
result.to_csv(
    "chapter04_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)
misclassified.to_csv(
    "chapter04_misclassified.csv",
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료")

## 프롬프트 예시

```
방금 저장한 chapter04_misclassified.csv가 제대로 저장됐는지
다시 읽어서 크기와 앞부분을 확인하고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
check = pd.read_csv("chapter04_misclassified.csv", encoding="utf-8-sig")
print(check.shape)
check.head()

---
# 실습 17. 새로운 도서 제목 예측하기

## 해야 할 일

학습에 없던 제목을 넣어 본다.

여기서 가장 중요한 점은 **새 제목에 `fit_transform()`을 쓰지 않는 것**이다.

> 학습된 `tfidf.transform()` -> 학습된 `model.predict()`

## 프롬프트 예시

```
학습된 tfidf와 model이 있습니다.
학습에 없던 새 제목 3개의 분야를 예측하고 싶습니다.
'파이썬으로 시작하는 데이터 분석', '처음 배우는 주식 투자', '마음을 이해하는 심리학'
새 제목에는 fit_transform이 아니라 transform만 써야 합니다.
제목과 예상 분야를 표로 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
new_titles = [
    "파이썬으로 시작하는 데이터 분석",
    "처음 배우는 주식 투자",
    "마음을 이해하는 심리학",
]

new_vectors = tfidf.transform(new_titles)
new_predictions = model.predict(new_vectors)

pd.DataFrame({
    "상품명": new_titles,
    "예상_분야": new_predictions,
})

### 예측 결과를 그대로 믿지 않는다

예측이 이상하게 나왔다면 **왜 그렇게 예측했는지** 확인한다.
각 제목에서 단어 사전에 있는 단어가 몇 개였는지, 분야별 확률이 어땠는지 본다.

## 프롬프트 예시

```
새 제목 3개를 예측했는데 결과가 이상한 것이 있습니다.
각 제목마다 TF-IDF 단어 사전에 있던 단어가 무엇인지,
그리고 model.predict_proba로 확률이 높은 분야 3개를 확률과 함께 출력해서
왜 그렇게 예측했는지 확인하고 싶습니다.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
feature_names = tfidf.get_feature_names_out()

for i, title in enumerate(new_titles):
    row = new_vectors.getrow(i)
    probs = model.predict_proba(row)[0]
    top3 = probs.argsort()[::-1][:3]

    print(f"[{title}]")
    print("   사전에 있던 단어:", feature_names[row.indices].tolist() or "없음")
    print("   확률 상위 3개:", [(model.classes_[k], round(probs[k], 3)) for k in top3])
    print()

### `파이썬으로 시작하는 데이터 분석`이 `소설`로 나온 이유

이 제목의 단어는 **train 단어 사전에 하나도 없었다.**

- 기본 TfidfVectorizer는 조사를 떼지 않아 `파이썬으로`, `시작하는`이 통째로 하나의 토큰이 된다.
- `데이터`, `분석`은 이번 베스트셀러 제목에 거의 쓰이지 않아 사전에 없다.

단어 정보가 0이면 Naive Bayes는 **각 분야의 원래 비율(사전 확률)** 만으로 판단한다.
train에서 가장 많은 분야가 `소설`이므로 `소설`로 예측한 것이다.
확률도 0.19로 낮아서, 모델 스스로도 **확신이 없는 상태**다.

> 예측 결과가 나왔다고 해서 모델이 그 제목을 이해한 것은 아니다.

---
# 보충. 모델이 무엇을 배웠고 어디서 틀렸는지 파고들기

## 해야 할 일

Accuracy와 report만으로는 **왜** 이 정도 성능이 나왔는지 설명하기 어렵다.
예측이 어디로 몰렸는지, 분야별로 무엇을 배웠는지, 데이터 정리 기준이 결과에 어떤 영향을 줬는지 확인한다.

## 프롬프트 예시

```
실제 분야 y_test와 예측 y_pred가 있습니다.
예측이 특정 분야로 몰렸는지 확인하고 싶습니다.
분야별 실제 건수와 예측 건수를 한 표에 놓고 실제 건수 순으로 정렬해 주세요.
예측에 한 번이라도 나온 분야가 몇 개인지도 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 1) 예측이 어느 분야로 몰렸는가 — 실제 분포와 비교
spread = pd.DataFrame({
    "실제_건수": pd.Series(y_test).value_counts(),
    "예측_건수": pd.Series(y_pred).value_counts(),
}).fillna(0).astype(int).sort_values("실제_건수", ascending=False)

print("예측에 한 번이라도 나온 분야:", int((spread["예측_건수"] > 0).sum()), "/", len(spread))
spread.head(12)

## 프롬프트 예시

```
분야별 precision, recall, F1을 train 데이터 수와 함께 한 표로 보고 싶습니다.
precision_recall_fscore_support를 쓰고, 분야 순서는 model.classes_로 맞춰 주세요.
train 데이터가 많은 순으로 정렬하고,
F1이 0보다 큰 분야가 몇 개인지도 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 2) 분야별 성능을 train 데이터 수와 함께 본다
from sklearn.metrics import precision_recall_fscore_support

prec, rec, f1, sup = precision_recall_fscore_support(
    y_test, y_pred, labels=model.classes_, zero_division=0
)

by_field = pd.DataFrame({
    "분야": model.classes_,
    "train수": [int((y_train == c).sum()) for c in model.classes_],
    "test수": sup,
    "precision": prec.round(2),
    "recall": rec.round(2),
    "F1": f1.round(2),
}).sort_values("train수", ascending=False).reset_index(drop=True)

print("F1이 0보다 큰 분야:", int((by_field["F1"] > 0).sum()), "/", len(by_field))
by_field.head(12)

## 프롬프트 예시

```
학습된 MultinomialNB model과 단어 목록 feature_names가 있습니다.
분야별로 모델이 가장 중요하게 본 단어 6개를 보고 싶습니다.
model.feature_log_prob_를 써서
소설, 취업/수험서, 외국어, 경제/경영, 어린이(초등), 만화를 확인해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 3) 분야별로 모델이 가장 중요하게 본 단어
import numpy as np

for field in ["소설", "취업/수험서", "외국어", "경제/경영", "어린이(초등)", "만화"]:
    idx = list(model.classes_).index(field)
    top = np.argsort(model.feature_log_prob_[idx])[::-1][:6]
    print(f"{field:10}:", ", ".join(feature_names[top]))

## 프롬프트 예시

```
데이터를 어떻게 정리하느냐에 따라 정확도가 어떻게 달라지는지 비교하고 싶습니다.
데이터를 넣으면 split -> TF-IDF -> NB 학습 -> 정확도와 기준선을 돌려주는
evaluate(data, label) 함수를 만들고,
'1권 분야만 제외'한 경우와 '1권 분야 + 미분류 제외'한 경우를 표로 비교해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 4) 데이터 정리 기준에 따라 결과가 어떻게 달라지는가
def evaluate(data, label):
    Xa, ya = data["상품명"], data["분야"]
    Xtr, Xte, ytr, yte = train_test_split(Xa, ya, test_size=0.2, random_state=42, stratify=ya)
    vec = TfidfVectorizer()
    nb = MultinomialNB().fit(vec.fit_transform(Xtr), ytr)
    acc = accuracy_score(yte, nb.predict(vec.transform(Xte)))
    base = (yte == ytr.value_counts().index[0]).mean()
    return {"기준": label, "도서 수": len(data), "분야 수": ya.nunique(),
            "Accuracy": round(acc, 4), "기준선": round(base, 4)}

raw = df_books[["상품명", "분야"]].dropna().copy()
counts = raw["분야"].value_counts()
only_rare_removed = raw[raw["분야"].isin(counts[counts >= 2].index)]

pd.DataFrame([
    evaluate(only_rare_removed, "1권 분야만 제외"),
    evaluate(df_model, "1권 분야 + 미분류 제외 (이번 기준)"),
])

### 보충 분석에서 확인한 것

**예측이 소설로 쏠렸다** — test 184권 중 135권(73%)을 `소설`로 예측했다.
모델은 31개 분야를 학습했고 test에는 27개 분야가 있었지만, **예측에 한 번이라도 쓰인 분야는 7개**뿐이었다.
`소설`의 recall은 1.00이지만 precision은 0.27이다.
"애매하면 소설"로 찍어서 소설은 다 맞히지만, 소설이라고 한 것의 대부분이 틀렸다는 뜻이다.

**잘 맞힌 분야는 전용 단어가 있는 분야였다** — `취업/수험서`와 `외국어`가 F1 0.86으로 가장 높았다.
모델이 중요하게 본 단어를 보면 이유가 분명하다.

```
취업/수험서 : 2027, 2026, 해커스경찰, 기본서, 공단기, 경찰학
외국어      : 토익, jlpt, 일본어, 일본어능력시험, 해커스, lc
```

이 단어들은 **다른 분야에서는 거의 쓰이지 않는다.** 그래서 제목만 봐도 분야가 드러난다.

**전용 단어가 없는 분야는 하나도 맞히지 못했다** — `시/에세이`(train 58권), `자기계발`(46권),
`만화`(33권)는 데이터가 적지 않은데도 F1이 0이었다.
이 분야 제목은 `싫어하는 사람을 사라지게 하는 방법`, `흔들리는 나에게, 고전이 말했다`처럼
**일반적인 단어로 된 문장형 제목**이라 소설과 구분되지 않는다.

> 데이터 수보다 **제목에 분야를 드러내는 단어가 있느냐**가 성능을 더 크게 좌우했다.

**미분류를 빼니 정확도가 올랐다** — 1권 분야만 뺐을 때 0.3724에서 미분류까지 빼면 0.4130이 됐다.
의미 없는 라벨을 정답에서 빼는 것만으로 결과가 달라진다.

---
# 보충 2. 소설 쏠림을 줄일 수 있을까 — 스무딩 값 `alpha`

## 해야 할 일

MultinomialNB에는 `alpha`(스무딩) 값이 있고 기본값은 **1.0**이다.

스무딩은 train에서 한 번도 못 본 단어에도 조금씩 점수를 나눠 주는 장치다.
그런데 도서 제목처럼 **짧고 단어가 거의 겹치지 않는 데이터**에서는
스무딩이 너무 강하면 단어가 주는 단서가 묻혀 버리고, 결국 **사전 확률(소설)** 이 이기게 된다.

`alpha`를 바꿔 보면서 이게 맞는지 확인한다.

### 중요 — alpha는 test를 보고 고르면 안 된다

test 정확도를 보면서 alpha를 고르면 **test가 학습 과정에 들어간 것**이 된다.
실습 7에서 배운 데이터 누수와 같은 문제다.

그래서 **train 안에서만 5번 나눠 검증(교차검증)** 해서 alpha를 고르고,
test는 마지막에 **딱 한 번만** 확인한다.
`Pipeline`으로 TF-IDF와 모델을 묶으면, 교차검증 때마다 TF-IDF도 train 조각에만 fit된다.

## 프롬프트 예시

```
MultinomialNB의 alpha 값을 바꿔 보고 싶습니다.
단, test를 보고 고르면 데이터 누수이므로 train 안에서만 교차검증으로 고르고 싶습니다.
TfidfVectorizer와 MultinomialNB를 Pipeline으로 묶고,
GridSearchCV로 alpha를 0.01, 0.05, 0.1, 0.3, 0.5, 1.0 중에서 골라 주세요.
교차검증은 KFold 5번(shuffle, random_state=42)으로 하고,
alpha별 교차검증 정확도를 표로 보여 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold

pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("nb", MultinomialNB()),
])

search = GridSearchCV(
    pipe,
    {"nb__alpha": [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]},
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy",
)
search.fit(X_train, y_train)   # train만 사용한다

cv_table = pd.DataFrame({
    "alpha": search.cv_results_["param_nb__alpha"],
    "train 교차검증 정확도": search.cv_results_["mean_test_score"].round(4),
})

print("train 교차검증으로 선택된 alpha:", search.best_params_["nb__alpha"])
cv_table

## 프롬프트 예시

```
교차검증으로 alpha를 고른 search 객체가 있습니다.
기본 모델(alpha=1.0)과 조정한 모델의 test 정확도,
소설로 예측한 수, 예측에 쓰인 분야 수를 나란히 비교하고,
새 제목들을 두 모델이 각각 어떻게 예측하는지도 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 선택된 alpha로 test를 한 번만 평가하고 기본값과 비교한다
best_pred = search.predict(X_test)

print(f"기본   alpha=1.0  test 정확도: {accuracy_score(y_test, y_pred):.4f}")
print(f"조정   alpha={search.best_params_['nb__alpha']:<4} test 정확도: {accuracy_score(y_test, best_pred):.4f}")
print()
print("소설로 예측한 수  :", int((y_pred == "소설").sum()), "->", int((best_pred == "소설").sum()))
print("예측에 쓰인 분야 수:", len(set(y_pred)), "->", len(set(best_pred)))
print()
print("새 제목 예측 (기본 -> 조정):")
for t in new_titles + ["The Scent of Page : 디퓨저 300ML"]:
    print(f"  {t:32} {model.predict(tfidf.transform([t]))[0]:>8}  ->  {search.predict([t])[0]}")

### 실험 결과

| 항목 | 기본 (alpha=1.0) | 조정 (alpha=0.01) |
|---|---|---|
| train 교차검증 정확도 | 0.3716 | 0.5041 |
| **test 정확도** | **0.4130** | **0.5380** |
| 소설로 예측한 수 | 135 | 67 |
| 예측에 쓰인 분야 수 | 7 | 19 |

스무딩을 약하게 하자 **소설 쏠림이 절반으로 줄고**, 예측에 쓰이는 분야가 7개에서 19개로 늘었다.
새 제목 예측도 바뀌었다.

```
마음을 이해하는 심리학          소설  ->  인문
The Scent of Page : 디퓨저 300ML  소설  ->  책향
```

즉 소설 쏠림의 상당 부분은 데이터 탓만이 아니라 **기본 설정이 이 데이터에 맞지 않았기 때문**이었다.

> 다만 이 결과도 184권짜리 test 한 번의 결과다.
> 이번 Chapter의 목표는 성능 올리기가 아니므로, 본 실습의 모델은 블로그대로 기본값을 유지했다.

---
# 실습 18. 모델의 한계 이해하기

현재 모델은 **도서 제목만** 사용한다. 다음 정보는 사용하지 않는다.

```
책 소개 / 목차 / 저자 / 출판사 / 키워드 / 본문 / 독자 리뷰
```

따라서 제목에 분야 정보가 충분하지 않으면 모델이 구분하기 어렵다.
또한 베스트셀러 데이터는 전체 출판 도서를 대표하지 않을 수 있다.

모델이 특정 분야로 예측했다고 해서 그것이 공식 분류라는 뜻도 아니다.

| 좋은 표현 | 피해야 할 표현 |
|---|---|
| 현재 학습 데이터의 패턴을 바탕으로 모델이 경제/경영 분야로 **예측했다.** | 이 책은 경제/경영 분야의 **책이다.** |

---
# 실습 19. 데이터 누수 최종 점검

- train/test를 TF-IDF fit보다 먼저 나눴는가?
- `fit_transform()`은 X_train에만 적용했는가?
- X_test에는 `transform()`만 적용했는가?
- 모델 학습에는 train 데이터만 사용했는가?
- 새 제목에도 기존 Vectorizer의 `transform()`을 사용했는가?

**이 다섯 가지를 지키는 것이 이번 Chapter의 핵심이다.**

## 프롬프트 예시

```
TF-IDF 단어 사전이 train만으로 만들어졌는지 숫자로 확인하고 싶습니다.
지금 tfidf의 단어 수, train만으로 fit했을 때 단어 수,
전체 X로 fit했을 때 단어 수를 비교해 주세요.
전체로 fit했다면 test에서 몇 개의 단어가 새로 들어왔을지도 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 단어 사전이 train만으로 만들어졌는지 숫자로 확인한다
train_only = TfidfVectorizer().fit(X_train)
all_data = TfidfVectorizer().fit(X)

print("지금 tfidf의 단어 수      :", len(tfidf.vocabulary_))
print("train만으로 fit한 단어 수 :", len(train_only.vocabulary_))
print("전체로 fit했다면 단어 수  :", len(all_data.vocabulary_))
print()
print("train에만 fit했는가?:", len(tfidf.vocabulary_) == len(train_only.vocabulary_))
print("전체로 fit했다면 test에서 새로 들어왔을 단어 수:",
      len(all_data.vocabulary_) - len(train_only.vocabulary_))

전체 데이터로 fit했다면 **test에만 있는 단어**가 사전에 들어간다.
모델이 시험 문제를 미리 본 것과 같아서, 평가 점수가 실제보다 좋게 나올 수 있다.

---
# 실습 20. 자주 만나는 오류

### stratify 오류

클래스에 데이터가 너무 적으면 오류가 난다. **이번 데이터에서 실제로 발생했다.**
무조건 `stratify=None`으로 바꾸기보다 작은 클래스의 데이터 수와 라벨을 먼저 확인한다.
(실습 4에서 1권뿐인 분야 6개를 확인하고 처리했다.)

### empty vocabulary

입력이 비어 있거나 전처리 후 사용할 단어가 없을 때 난다.

### X와 y 길이 불일치

같은 `df_model`에서 X와 y를 만드는 것이 안전하다.

### 새 제목에서 다시 fit하는 실수

```python
tfidf.fit_transform(["새로운 도서 제목"])   # 잘못된 코드
tfidf.transform(["새로운 도서 제목"])       # 올바른 코드
```

## 프롬프트 예시

```
텍스트 분류에서 자주 나는 오류를 미리 점검하고 싶습니다.
가장 적은 분야의 도서 수, X_train 개수와 앞 5개,
X와 y의 길이가 같은지를 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
# 위 오류들을 미리 점검한다
print("가장 적은 분야의 도서 수 (2 이상이어야 stratify 가능):", y.value_counts().min())
print()
print("X_train 개수:", len(X_train))
print("X_train 앞 5개:", X_train.head(5).tolist())
print()
print("X 길이:", len(X), "/ y 길이:", len(y), "/ 같은가?:", len(X) == len(y))

---
# 실습 21. 전체 코드 흐름

코드를 외우기보다 다음 단계 이름으로 읽어 본다.

> 데이터 -> X/y -> split -> TF-IDF -> fit -> predict -> evaluate -> new prediction

> 블로그의 정리 코드를 그대로 쓰면 이번 데이터에서는 `stratify` 오류가 난다.
> 그래서 **1권뿐인 분야와 미분류를 빼는 줄**을 추가했다.

## 프롬프트 예시

```
입력 X와 정답 y가 있습니다.
train_test_split으로 20%를 test로 나누고 싶습니다.
같은 결과가 나오게 random_state=42를 쓰고,
분야 비율이 train과 test에서 비슷하도록 stratify=y를 써 주세요.
train과 test의 크기를 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 1. 데이터
df = pd.read_csv("book_bestseller_clean.csv", encoding="utf-8-sig")
df_model = df[["상품명", "분야"]].dropna().copy()
df_model["상품명"] = df_model["상품명"].astype(str).str.strip()
df_model["분야"] = df_model["분야"].astype(str).str.strip()
df_model = df_model[
    (df_model["상품명"] != "") &
    (df_model["분야"] != "")
]

# 1-1. stratify를 위해 1권뿐인 분야와 미분류를 뺀다 (이번 데이터에서 추가)
counts = df_model["분야"].value_counts()
df_model = df_model[
    df_model["분야"].isin(counts[counts >= 2].index) &
    (df_model["분야"] != "미분류")
]

# 2. X, y
X = df_model["상품명"]
y = df_model["분야"]

# 3. train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 4. TF-IDF
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 5. 학습
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# 6. 평가
y_pred = model.predict(X_test_tfidf)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

# 7. 새 제목 예측
new_title = ["처음 배우는 주식 투자"]
new_vector = tfidf.transform(new_title)
print("예상 분야:", model.predict(new_vector)[0])

---
# 실습 22. 결과 정리

> 아래 숫자는 위 셀들의 **실제 실행 결과**를 보고 적은 것이다.

## Chapter 04 결과

### 데이터
- 입력 X: 상품명 / 정답 y: 분야
- 원본 986권 중 **1권뿐인 분야 6개**와 **미분류 64권**을 제외해 **916권, 31개 분야** 사용
  - 1권뿐인 분야는 `stratify=y`로 나눌 수 없어 제외했다 (강연·가요·필기구는 비도서 상품)
  - 미분류는 실제 분야가 아니라 Chapter 01에서 결측치를 채운 값이라 제외했다
- Train/Test: 80/20 (732권 / 184권), `stratify=y`

### 모델
- TfidfVectorizer (기본 설정, train에만 fit, 단어 1,807개)
- MultinomialNB (기본 설정)

### 평가
- **Accuracy: 0.4130** (184권 중 76권 정답)
- train 정확도는 0.6011로, 공부한 문제도 40%를 틀렸다
  -> 외워서 생긴 문제(과적합)보다 **제목에 분야 단서가 부족한 것**이 주된 원인
- 가장 많은 분야(소설)만 찍었을 때의 기준선 0.1957보다 **약 2.1배** 높다
- 분야별 성능 차이가 컸다

| 분야 | train수 | precision | recall | F1 |
|---|---|---|---|---|
| 취업/수험서 | 63 | 0.79 | 0.94 | **0.86** |
| 외국어 | 46 | 1.00 | 0.75 | **0.86** |
| 소설 | 142 | 0.27 | 1.00 | 0.42 |
| 시/에세이 | 58 | 0.00 | 0.00 | **0.00** |
| 자기계발 | 46 | 0.00 | 0.00 | **0.00** |

### 오분류
- 오분류 108건 중 **99건(92%)이 소설로 예측**됐다
- 원인별: 비도서 상품 5건 / 사전 단어 0개 25건 / 단서 단어 1~2개 56건 / 다른 분야로 끌림 22건
  (시/에세이 -> 소설 14건, 인문 -> 소설 13건, 어린이(초등) -> 소설 12건, 경제/경영 -> 소설 11건)
- 대표 사례
  - `인플레이션은 누구를 부자로 만드는가` (경제/경영 -> 소설)
    : 경제 전용 단어인 `부자`가 `부자로`처럼 조사와 붙어 있어 사전의 `부자`와 다른 토큰이 됐다
  - `미움받을 용기(200만 부 기념 스페셜 에디션)` (인문 -> 소설)
    : `에디션`은 train에서 소설에 자주 나온 단어라 소설 쪽으로 끌려갔다
  - `[비욘드오리진] ABC착즙액 (50ml*20포)` (홈/라이프 -> 소설)
    : 도서가 아닌 상품이라 제목만으로는 분야를 알 수 없다

### 모델이 배운 것
- 잘 맞힌 분야는 **그 분야에서만 쓰는 단어**가 있었다
  (수험서: 해커스경찰·공단기·기본서 / 외국어: 토익·jlpt·일본어능력시험)
- 못 맞힌 분야는 **일반적인 단어로 된 문장형 제목**이라 소설과 구분되지 않았다
- 즉 **데이터 수보다 제목에 분야를 드러내는 단어가 있는지**가 성능을 좌우했다

### 한계
- 제목만 사용했다. 책 소개·목차·저자 정보가 없다
- 기본 TfidfVectorizer가 조사를 떼지 않아 `부자`와 `부자로`를 다른 단어로 본다
  (Chapter 02의 Kiwi 형태소 분석을 적용하면 개선될 여지가 있다)
- test가 184권뿐이라, 분야별로는 1~5권으로 평가한 곳이 많다. 분야별 수치는 흔들림이 크다
- 현재 베스트셀러 데이터와 라벨 범위 안에서의 평가다
- 새 제목 `파이썬으로 시작하는 데이터 분석`은 사전에 있는 단어가 0개여서
  사전 확률만으로 `소설`이 나왔다. **예측이 나왔다고 해서 모델이 제목을 이해한 것은 아니다**

---
# 핵심 시사점

### 1. 제목만으로 분야를 맞히는 데에는 분명한 한계가 있다
기본 모델의 test 정확도는 0.4130, train 정확도도 0.6011에 그쳤다.
공부한 문제도 40%를 틀린다는 것은 모델이 외우지 못한 게 아니라
**제목 안에 분야를 가려낼 단서가 부족하다**는 뜻이다.
사전에 있는 단어가 3개 이상인 제목은 정확도가 0.613이었지만, 1~2개인 제목은 0.3 이하였다.

### 2. 잘 맞히는 분야는 "그 분야 전용 단어"가 있는 분야다
`취업/수험서`(F1 0.86)와 `외국어`(F1 0.86)는 `공단기`, `해커스경찰`, `토익`, `jlpt`처럼
**다른 분야에서는 쓰지 않는 단어**가 제목에 들어간다.
반면 `시/에세이`, `자기계발`은 58권, 46권으로 적지 않은데도 F1이 0이었다.
**데이터 양보다 제목에 분야가 드러나느냐가 성능을 좌우했다.**

### 3. 모델은 모르면 "가장 흔한 분야"로 찍는다
틀린 108건 중 99건(92%)이 소설로 예측되었다.
단서가 약하면 Naive Bayes는 사전 확률에 기대고, train에서 가장 많은 분야가 소설이기 때문이다.
**Accuracy만 보면 이 쏠림이 보이지 않는다.** 분야별 지표와 예측 분포를 함께 봐야 한다.

### 4. 한국어 조사 처리가 가장 큰 구조적 원인이다
`부자로`와 `부자`를 다른 단어로 보는 바람에, 사전에 단어가 있어도 못 쓰는 제목이 25건이었다.
Chapter 02의 Kiwi 형태소 분석처럼 **명사만 뽑아서 넣으면** 개선될 여지가 크다.

### 5. 기본 설정이 데이터에 맞는지 확인해야 한다
스무딩 값 alpha만 1.0에서 0.01로 바꿔도 test 정확도가 0.4130에서 0.5380으로 올랐고,
소설 쏠림은 135건에서 67건으로 줄었다.
단, 이 값은 **train 안의 교차검증으로 골라야** 한다. test를 보고 고르면 그것도 데이터 누수다.

### 6. 데이터 정리가 모델 결과를 바꾼다
`미분류` 64권을 정답에서 빼자 정확도가 0.3724에서 0.4130으로 올랐다.
디퓨저·담요 같은 **비도서 상품**은 여전히 섞여 있어 오분류 원인이 되었다.
Chapter 01에서 발견한 데이터 문제가 Chapter 04의 결과에 그대로 이어진다.

## 개선 방향

| 방향 | 기대 효과 | 근거 |
|---|---|---|
| Kiwi로 명사만 추출해서 입력 | 조사 문제 해결 | 사전 단어 0개 25건 |
| alpha를 교차검증으로 조정 | 소설 쏠림 감소 | 0.4130 -> 0.5380 |
| 비도서 상품 제외 | 의미 없는 오분류 제거 | 원인 1번 5건 |
| 판매용 문구(에디션·기념·스페셜)를 불용어로 | 엉뚱한 분야로 끌리는 것 방지 | 원인 4번 |
| 비슷한 분야 통합 (문학+소설 등) | 경계가 모호한 분야 정리 | `문학 -> 소설` 오분류 |
| 책 소개·저자 등 제목 외 정보 추가 | 단서 부족 해결 | 원인 3번 56건 |

> 이번 결과는 986권짜리 하루치 베스트셀러, test 184권 기준이다.
> 분야별로는 1~5권으로 평가한 곳이 많아 수치의 흔들림이 크다는 점을 함께 기억한다.

---
# 최종 체크리스트

- [x] 상품명과 분야 데이터를 정리했다.
- [x] X와 y의 역할을 설명할 수 있다.
- [x] train/test를 먼저 분리했다.
- [x] TF-IDF는 train에만 fit했다.
- [x] MultinomialNB를 train 데이터로 학습했다.
- [x] test 데이터로 성능을 확인했다.
- [x] accuracy뿐 아니라 분야별 지표를 확인했다.
- [x] 오분류 제목을 직접 확인했다.
- [x] 새 제목에는 transform만 사용했다.
- [x] 모델의 한계를 설명할 수 있다.

---
# 이번 Chapter에서 꼭 기억할 5가지

1. X는 도서 제목, y는 분야다.
2. train은 학습용, test는 평가용이다.
3. **TF-IDF도 학습 과정이므로 train에만 fit한다.**
4. Accuracy 하나보다 분야별 지표와 오분류를 함께 본다.
5. 새 데이터에서는 다시 fit하지 않고 **transform 후 predict**한다.

---
# 다음 Chapter 연결

```
Chapter 03   도서 제목 -> TF-IDF 벡터
Chapter 04   TF-IDF 벡터 -> Naive Bayes -> 도서 분야 예측
Chapter 05   TF-IDF 벡터 -> Cosine Similarity -> 비슷한 도서 찾기 -> Top 5 추천
```

같은 벡터가 **분류와 추천이라는 서로 다른 문제**에 활용된다는 점을 연결해서 이해한다.

## Chapter 04 한 문장 정리

> 원본 데이터를 먼저 train/test로 나누고, TF-IDF를 train에만 fit한 뒤
> Multinomial Naive Bayes로 학습하고, test 성능과 오분류를 함께 검증한다.